## Imports and Setup

In [5]:
# Add project root to Python path
import sys
from pathlib import Path

DIR_ROOT = str(Path().resolve().parent)
sys.path.insert(0, str(Path().resolve().parent))
%reload_ext autoreload
%autoreload 2

import json
import serpapi
import re
import os
from config import API_KEY_SERPAPI

DIR_DATA = os.path.join(DIR_ROOT, "data")
DIR_DATA_OUT = os.path.join(DIR_DATA, "out")
DIR_JSON_DATA = os.path.join(DIR_DATA_OUT, "json")


## Setup the api call and call the search API

In [6]:
client = serpapi.Client(api_key=API_KEY_SERPAPI)
results = client.search({
  "engine": "google_scholar_author",
  "author_id": "T48JalQAAAAJ",
  "sort": "pubdate",
  "num": "100"
})

In [ ]:
results["articles"][0]

## Loop through the articles and decide if the target author is first, last or middle in the autheor list

In [ ]:
articles = results["articles"]
for article in articles:
    authors = article["authors"]
    authors_list = authors.split(",")
    author_position = "middle"
    if re.search("(?i)maynard", authors_list[0]):
        author_position = "first"
    elif re.search("(?i)maynard", authors_list[-1]):
        author_position = "last"
    article["author_position"] = author_position

print(f"Number of articles {len(articles)}")
print(f"Number where maynard is first {len([_ for _ in articles if _['author_position'] == 'first'])}")
print(f"Number where maynard is last {len([_ for _ in articles if _['author_position'] == 'last'])}")

In [ ]:
for article in articles:
    if article["author_position"] == "last":
        print(article["link"])

In [ ]:
# Fake the data object
data = {
    "T48JalQAAAAJ" : {
        "author_url": "https://scholar.google.com/citations?user=T48JalQAAAAJ&hl=en"
    }
}

# Check for existing json data
json_data_files = [
    f for f 
    in os.listdir(DIR_JSON_DATA) 
    if os.path.isfile(os.path.join(DIR_JSON_DATA, f)) and ".json" in f]

for json_data_file in json_data_files:
    author_id = json_data_file.replace(".json", "")
    if author_id in data:
        with open(os.path.join(DIR_JSON_DATA, json_data_file), "r") as f:
            data[author_id]["json_data"] = json.load(f)

# print(data[author_id]["json_data"].keys())


In [ ]:
# Add the reponse we have and write out the files
data["T48JalQAAAAJ"]["json_data"] = results.as_dict()

for author_id, d in data.items():
    author_json_data_file_path = os.path.join(DIR_JSON_DATA, f"{author_id}.json")
    with open(author_json_data_file_path, "w") as f:
        json.dump(d["json_data"], f, indent=4)